## Data Science Project - Flight Price Prediction  2/2



### Step 5 – Model Building

In [2]:
import os

In [3]:
# 1 Importation des librairies
import pandas as pd
from sklearn.model_selection import train_test_split

# Modèles ML
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor

#2 Charger le dataset encodé et scalé
df = pd.read_csv("Clean_Dataset_scaled.csv")

# Supprimer la colonne 'flight' si elle existe
df = df.drop(columns=['flight'], errors='ignore')

# 3 Separer features et target
X = df.drop('price', axis=1)
y = df['price']

# 4 Split
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

# 5 Définition des modèles 
models = {
    'LinearRegression': LinearRegression(),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(random_state=42),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'KNN': KNeighborsRegressor(),
    #les 2 modèles que nous avons choisis : Adaboost & Extra Trees Regressor
    'AdaBoost': AdaBoostRegressor(random_state=42),
    'ExtraTrees': ExtraTreesRegressor(random_state=42),
}

# 6 Entraînement
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train) # entrainer le modèle sur les données d'entrainement
    trained_models[name] = model    # stocker le modèle entrainé dans un nouveau dictionnaire
    print(f"{name} entraîné avec succès")

print(" Tous les modèles ont été entraînés !")

LinearRegression entraîné avec succès
DecisionTree entraîné avec succès
RandomForest entraîné avec succès
GradientBoosting entraîné avec succès
KNN entraîné avec succès
AdaBoost entraîné avec succès
ExtraTrees entraîné avec succès
 Tous les modèles ont été entraînés !


### Step 6 – Model Evaluation

In [4]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
import pandas as pd
import numpy as np

results = []

print("\n Évaluation des modèles avec Cross-Validation...\n")

for name, model in trained_models.items():
    # Prédictions
    y_pred = model.predict(X_test)

    # Métriques sur test set
    mae  = mean_absolute_error(y_test, y_pred)
    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_test, y_pred)
    # Cross-Validation (k=5)
    cv_score = cross_val_score(model, X, y, cv=5, scoring='r2').mean()

    # Stockage
    results.append({
        'Modèle': name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2 Score': r2,
        'CV Score (5-fold)': cv_score
    })

    print(f" {name} — OK")

# Convertir en DataFrame
results_df = pd.DataFrame(results)

print("\n Résultats finaux :")
results_df



 Évaluation des modèles avec Cross-Validation...

 LinearRegression — OK
 DecisionTree — OK
 RandomForest — OK
 GradientBoosting — OK
 KNN — OK
 AdaBoost — OK
 ExtraTrees — OK

 Résultats finaux :


,Modèle,MAE,MSE,RMSE,R2 Score,CV Score (5-fold)
0,LinearRegression,0.200605,0.088746,0.297902,0.911305,-0.295270
1,DecisionTree,0.051999,0.024417,0.156261,0.975597,0.332663
2,RandomForest,0.047843,0.015224,0.123386,0.984785,0.558542
3,GradientBoosting,0.130940,0.048548,0.220335,0.951480,0.512328
4,KNN,0.081621,0.029450,0.171610,0.970567,-27.314168
5,AdaBoost,0.183074,0.076448,0.276493,0.923595,0.498657
6,ExtraTrees,0.049793,0.017166,0.131021,0.982843,0.518177


### Step 7– Save the Final Model

In [5]:
import pickle

# Trouver le meilleur modèle selon R2
best_row = results_df.sort_values(by='R2 Score', ascending=False).iloc[0] #trouver la ligne du modèle ayant le meilleur R2 dans le dataframe des résultats
best_model_name = best_row['Modèle'] # récupérer le nom du meilleur modèle
best_model = trained_models[best_model_name] # récupérer l'objet du modèle entrainé correspondant au meilleur modèle

# Sauvegarde
pickle.dump(best_model, open('flight_price_model.pkl', 'wb'))

# afficher le nom du meilleur modèle
print(f"Meilleur modèle : {best_model_name}")
print("Modèle sauvegardé sous flight_price_model.pkl")

Meilleur modèle : RandomForest
Modèle sauvegardé sous flight_price_model.pkl


### Step 8 – Deploy the Model using FastAPI

In [6]:
import pickle

# Charger le modèle sauvegardé
loaded_model = pickle.load(open("flight_price_model.pkl", "rb"))

# Tester une prédiction
loaded_model.predict(X_test[:5])

array([-0.59857343,  2.22209826, -0.64740555,  1.74415478, -0.65281004])

In [7]:
import joblib

model = joblib.load("flight_price_model.pkl")
print(model.n_features_in_)

30


In [8]:
print(X_train.columns.tolist())


['duration', 'days_left', 'airline_Air_India', 'airline_GO_FIRST', 'airline_Indigo', 'airline_SpiceJet', 'airline_Vistara', 'source_city_Chennai', 'source_city_Delhi', 'source_city_Hyderabad', 'source_city_Kolkata', 'source_city_Mumbai', 'departure_time_Early_Morning', 'departure_time_Evening', 'departure_time_Late_Night', 'departure_time_Morning', 'departure_time_Night', 'stops_two_or_more', 'stops_zero', 'arrival_time_Early_Morning', 'arrival_time_Evening', 'arrival_time_Late_Night', 'arrival_time_Morning', 'arrival_time_Night', 'destination_city_Chennai', 'destination_city_Delhi', 'destination_city_Hyderabad', 'destination_city_Kolkata', 'destination_city_Mumbai', 'class_Economy']


In [9]:
#Extraire les listes uniques pour chaque feature catégorielle
df = pd.read_csv("../DataSet/Clean_Dataset.csv")
airlines = df['airline'].unique().tolist()
source_cities = df['source_city'].unique().tolist()
departure_times = df['departure_time'].unique().tolist()
stops_list = df['stops'].unique().tolist()
arrival_times = df['arrival_time'].unique().tolist()
destination_cities = df['destination_city'].unique().tolist()
classes = df['class'].unique().tolist()

print("Airlines:", airlines)
print("Source cities:", source_cities)
print("Departure times:", departure_times)
print("Stops:", stops_list)
print("Arrival times:", arrival_times)
print("Destination cities:", destination_cities)
print("Classes:", classes)


Airlines: ['SpiceJet', 'AirAsia', 'Vistara', 'GO_FIRST', 'Indigo', 'Air_India']
Source cities: ['Delhi', 'Mumbai', 'Bangalore', 'Kolkata', 'Hyderabad', 'Chennai']
Departure times: ['Evening', 'Early_Morning', 'Morning', 'Afternoon', 'Night', 'Late_Night']
Stops: ['zero', 'one', 'two_or_more']
Arrival times: ['Night', 'Morning', 'Early_Morning', 'Afternoon', 'Evening', 'Late_Night']
Destination cities: ['Mumbai', 'Bangalore', 'Kolkata', 'Hyderabad', 'Chennai', 'Delhi']
Classes: ['Economy', 'Business']
